# Pilot: one set of translations, four oracles

Pilot of [llm-code-migration-oracles](https://github.com/Nov17th/llm-code-migration-oracles). The code of the
pilot is written out below, so the notebook needs nothing else. Requires a GPU accelerator and Internet access.
Set `LIMIT = 5` for a quick test; leave `None` for the full run. The results are saved as a zip in the output.

In [1]:
LIMIT = None
TRANSLATOR = 'qwen2.5-coder:7b-instruct-q4_K_M'
JUDGE = 'llama3.1:8b-instruct-q4_K_M'
OLLAMA_VERSION = '0.40.11'
CODE = '/kaggle/working/pilot'
OUT = '/kaggle/working/results' if LIMIT is None else '/kaggle/working/results_test'

## Setup

In [2]:
import os, subprocess, time, urllib.request
os.makedirs(CODE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!which javac zstd > /dev/null || (apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless zstd > /dev/null)
!pip install -q codebleu==0.7.0 tree-sitter==0.22.3 tree-sitter-java==0.21.0
!curl -fsSL "https://ollama.com/download/ollama-linux-amd64.tar.zst?version={OLLAMA_VERSION}" | zstd -d | tar -x -C /usr
server = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/version'); break
    except Exception:
        time.sleep(2)
for model in (TRANSLATOR, JUDGE):
    subprocess.run(['ollama', 'pull', model], check=True, stdout=subprocess.DEVNULL)

Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 60e05f210007:   0% ▕                  ▏ 2.1 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   2% ▕                  ▏  77 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   3% ▕                  ▏ 126 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   4% ▕                  ▏ 206 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   6% ▕█                 ▏ 294 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   7% ▕█                 ▏ 342 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:   9% ▕█                 ▏ 419 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:  10% ▕█                 ▏ 489 MB/4.7 GB                  pulling manifest 
pulling 60e05f210007:  12% ▕██     

## Code (the same files as `pilot/` in the repository)

In [3]:
%%writefile /kaggle/working/pilot/data.py
"""Load the 164 HumanEval-X tasks, pairing each Python function with its Java counterpart."""
import gzip
import hashlib
import json
import os
import urllib.request
from dataclasses import dataclass

URL = 'https://raw.githubusercontent.com/THUDM/CodeGeeX/main/codegeex/benchmark/humaneval-x/{0}/data/humaneval_{0}.jsonl.gz'
SHA256 = {
    'python': '5752fb122efddb0e9b47c77907ae6ce9a15aac4dea14e44c46032c831e6b987a',
    'java': '276f36db2eb7aeb864f36a19a2f4e1bb2460fb6e22ff100f0b5bedabe3799967',
}


@dataclass
class Task:
    task_id: str
    python_code: str     # the function to translate, without its docstring
    java_signature: str  # imports, class Solution and the method signature
    java_imports: str
    java_reference: str  # signature + canonical solution
    java_body: str       # canonical solution only
    java_test: str       # class Main, which calls the method and checks the results


def _load(lang, data_dir):
    path = os.path.join(data_dir, 'humaneval_%s.jsonl.gz' % lang)
    if not os.path.exists(path):
        os.makedirs(data_dir, exist_ok=True)
        urllib.request.urlretrieve(URL.format(lang), path)
    with open(path, 'rb') as f:
        if hashlib.sha256(f.read()).hexdigest() != SHA256[lang]:
            raise RuntimeError('Unexpected content in ' + path)
    with gzip.open(path, 'rt', encoding='utf-8') as f:
        return [json.loads(line) for line in f]


def load_tasks(data_dir='data', limit=None):
    tasks = []
    for p, j in zip(_load('python', data_dir), _load('java', data_dir)):
        n = p['task_id'].split('/')[1]
        assert j['task_id'] == 'Java/' + n
        tasks.append(Task(
            task_id='HumanEval/' + n,
            python_code=p['declaration'] + p['canonical_solution'],
            java_signature=j['declaration'].rstrip() + '\n',
            java_imports=''.join(l + '\n' for l in j['declaration'].splitlines() if l.startswith('import ')),
            java_reference=j['declaration'] + j['canonical_solution'],
            java_body=j['canonical_solution'],
            java_test=j['test'],
        ))
    return tasks[:limit] if limit else tasks

Writing /kaggle/working/pilot/data.py


In [4]:
%%writefile /kaggle/working/pilot/llm.py
"""Talking to the models: the two prompts, the Ollama client, and parsing the answers."""
import hashlib
import json
import os
import re
import urllib.request

TRANSLATE_SYSTEM = 'You are an expert software engineer who translates code between programming languages.'
TRANSLATE_PROMPT = """Translate the following Python function into Java.

Python:
```python
{python}
```

The Java code must use exactly this class and method signature:
```java
{signature}```

Return the complete Java class, including the import lines, in a single ```java code block. Do not add a main method or any explanation."""

JUDGE_SYSTEM = 'You are an expert software engineer who reviews code translations.'
JUDGE_PROMPT = """Below is a Python function and a Java translation of it. Decide whether the Java method is functionally equivalent to the Python function: for every valid input, it must return the same result.

Python:
```python
{python}
```

Java:
```java
{java}
```

Answer with a single word on the first line, YES or NO, and then give a one-sentence reason."""

SEED = 1234


def translate_messages(task):
    return [{'role': 'system', 'content': TRANSLATE_SYSTEM},
            {'role': 'user', 'content': TRANSLATE_PROMPT.format(python=task.python_code.rstrip(), signature=task.java_signature)}]


def judge_messages(task, java_code):
    return [{'role': 'system', 'content': JUDGE_SYSTEM},
            {'role': 'user', 'content': JUDGE_PROMPT.format(python=task.python_code.rstrip(), java=java_code.rstrip())}]


def prompt_hashes():
    h = lambda s: hashlib.sha256(s.encode()).hexdigest()[:16]
    return {'translate': h(TRANSLATE_SYSTEM + TRANSLATE_PROMPT), 'judge': h(JUDGE_SYSTEM + JUDGE_PROMPT)}


class Ollama:
    """A model served by Ollama, queried with greedy decoding (temperature 0) and a fixed seed."""

    def __init__(self, model):
        self.model = model
        self.host = 'http://' + os.environ.get('OLLAMA_HOST', '127.0.0.1:11434').replace('http://', '')

    def _call(self, path, payload=None):
        data = json.dumps(payload).encode() if payload is not None else None
        req = urllib.request.Request(self.host + path, data=data, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=600) as r:
            return json.loads(r.read().decode())

    def chat(self, messages, max_tokens):
        options = {'temperature': 0, 'seed': SEED, 'num_ctx': 4096, 'num_predict': max_tokens}
        out = self._call('/api/chat', {'model': self.model, 'messages': messages, 'stream': False, 'options': options})
        return out['message']['content']

    def info(self):
        models = {m['name']: m for m in self._call('/api/tags')['models']}
        return {'model': self.model, 'digest': models.get(self.model, {}).get('digest'),
                'ollama': self._call('/api/version')['version'], 'temperature': 0, 'seed': SEED}


def extract_java(answer):
    """The first ```java block (else the first code block, else the whole answer), without package lines."""
    blocks = re.findall(r'```[ \t]*(\w*)[ \t]*\n(.*?)```', answer or '', re.S)
    java = [body for lang, body in blocks if lang.lower() == 'java']
    code = java[0] if java else (blocks[0][1] if blocks else answer or '')
    return '\n'.join(l for l in code.splitlines() if not re.match(r'\s*package\s+[\w.]+\s*;', l)).strip() + '\n'


def parse_verdict(answer):
    """YES or NO from the first non-empty line; else the first YES or NO anywhere; else UNPARSED."""
    text = (answer or '').strip()
    first_line = next((l for l in text.splitlines() if re.sub(r'[^A-Za-z]', '', l)), '')
    first_word = re.sub(r'[^A-Za-z ]', ' ', first_line).split()
    if first_word and first_word[0].upper() in ('YES', 'NO'):
        return first_word[0].upper()
    m = re.search(r'\b(YES|NO)\b', text.upper())
    return m.group(1) if m else 'UNPARSED'

Writing /kaggle/working/pilot/llm.py


In [5]:
%%writefile /kaggle/working/pilot/oracles.py
"""The four oracles. Each takes a task and a Java translation and returns its verdict as a dict."""
import os
import re
import shutil
import subprocess
import tempfile
import warnings

from llm import judge_messages, parse_verdict

COMPILE_TIMEOUT = 60
TEST_TIMEOUT = 10
THRESHOLDS = (0.5, 0.6, 0.7)


def _java(files, run_main=False):
    """Write the files to a fresh folder, compile them, and optionally run Main.
    Returns (status, log) with status in: ok, compile_error, fail, timeout."""
    d = tempfile.mkdtemp()
    try:
        for name, text in files.items():
            with open(os.path.join(d, name), 'w', encoding='utf-8') as f:
                f.write(text)
        steps = [(['javac', '-encoding', 'UTF-8', '-nowarn'] + list(files), COMPILE_TIMEOUT, 'compile_error')]
        if run_main:
            steps.append((['java', '-cp', '.', 'Main'], TEST_TIMEOUT, 'fail'))
        for cmd, timeout, error in steps:
            try:
                p = subprocess.run(cmd, cwd=d, capture_output=True, text=True, timeout=timeout)
            except subprocess.TimeoutExpired:
                return 'timeout', ''
            if p.returncode != 0:
                log = '\n'.join(l for l in (p.stdout + p.stderr).splitlines() if not l.startswith('Picked up '))
                return error, log[:2000]
        return 'ok', ''
    finally:
        shutil.rmtree(d, ignore_errors=True)


def compile_check(task, java_code):
    """O-b: the translation compiles on its own (the task's imports are put in front)."""
    status, log = _java({'Solution.java': task.java_imports + java_code})
    return {'compiles': status == 'ok', 'compile_log': log}


def test_check(task, java_code):
    """O-c: the translation passes the HumanEval-X tests. Status: pass, fail, timeout or compile_error.
    The tests and the translation share one file, which may hold only one public class (Main)."""
    code = re.sub(r'(^|\n)(\s*)public\s+(class\s+Solution\b)', r'\1\2\3', java_code)
    status, log = _java({'Main.java': task.java_imports + code + '\n' + task.java_test}, run_main=True)
    return {'tests': 'pass' if status == 'ok' else status, 'tests_log': log}


def similarity(task, java_code):
    """O-a: CodeBLEU against the reference solution, with its four components."""
    from codebleu import calc_codebleu
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        s = calc_codebleu([task.java_reference], [java_code], lang='java')
    return {'codebleu': round(s['codebleu'], 4), 'cb_ngram': round(s['ngram_match_score'], 4),
            'cb_weighted_ngram': round(s['weighted_ngram_match_score'], 4),
            'cb_syntax': round(s['syntax_match_score'], 4), 'cb_dataflow': round(s['dataflow_match_score'], 4)}


def judge(model, task, java_code):
    """O-h: a second model answers YES or NO to whether the translation is equivalent."""
    answer = model.chat(judge_messages(task, java_code), max_tokens=128)
    return {'judge': parse_verdict(answer), 'judge_answer': answer.strip()[:1000]}

Writing /kaggle/working/pilot/oracles.py


In [6]:
%%writefile /kaggle/working/pilot/pipeline.py
"""The stages of a run. Each stage appends one JSON line per item to a file in the output folder and skips
items already there, so an interrupted run continues where it stopped."""
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor

from llm import extract_java, translate_messages
from oracles import compile_check, judge, similarity, test_check


def read_jsonl(path):
    if not os.path.exists(path):
        return []
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]


def _todo(items, path, key):
    done = {r[key] for r in read_jsonl(path)}
    return [it for it in items if it[key] not in done]


def _save_each(path, items, work, workers=1):
    """Apply work(item) to each item (in parallel threads if workers > 1) and append item + result to path."""
    with ThreadPoolExecutor(max_workers=workers) as pool, open(path, 'a', encoding='utf-8') as f:
        for i, (it, res) in enumerate(zip(items, pool.map(work, items)), 1):
            f.write(json.dumps(dict(it, **res), ensure_ascii=False) + '\n')
            f.flush()
            if i % 20 == 0 or i == len(items):
                print('  %d/%d' % (i, len(items)), flush=True)


def _static_oracles(task, code):
    return dict(compile_check(task, code), **dict(test_check(task, code), **similarity(task, code)))


def translate(model, tasks, out):
    """Stage 1: one translation per task."""
    path = os.path.join(out, 'translations.jsonl')
    items = _todo([{'task_id': t.task_id} for t in tasks], path, 'task_id')
    by_id = {t.task_id: t for t in tasks}

    def work(it):
        start = time.time()
        answer = model.chat(translate_messages(by_id[it['task_id']]), max_tokens=1024)
        return {'answer': answer, 'java_code': extract_java(answer), 'seconds': round(time.time() - start, 1)}

    print('Translating %d tasks with %s' % (len(items), model.model))
    _save_each(path, items, work)


def check(tasks, out, workers):
    """Stage 2: compilation (O-b), tests (O-c) and CodeBLEU (O-a) for every translation."""
    path = os.path.join(out, 'oracles.jsonl')
    code = {r['task_id']: r['java_code'] for r in read_jsonl(os.path.join(out, 'translations.jsonl'))}
    by_id = {t.task_id: t for t in tasks}
    items = _todo([{'task_id': i} for i in by_id if i in code], path, 'task_id')
    print('Compile, tests and CodeBLEU for %d translations' % len(items))
    _save_each(path, items, lambda it: _static_oracles(by_id[it['task_id']], code[it['task_id']]), workers)


def judge_all(model, tasks, out):
    """Stage 3: the LLM judge (O-h) on every translation."""
    path = os.path.join(out, 'judgments.jsonl')
    code = {r['task_id']: r['java_code'] for r in read_jsonl(os.path.join(out, 'translations.jsonl'))}
    by_id = {t.task_id: t for t in tasks}
    items = _todo([{'task_id': i} for i in by_id if i in code], path, 'task_id')
    print('Judging %d translations with %s' % (len(items), model.model))
    _save_each(path, items, lambda it: judge(model, by_id[it['task_id']], code[it['task_id']]))


MUTATIONS = [  # one mutant per group: the first rule that applies, at its first position
    ('relational', [(' < ', ' <= '), (' <= ', ' < '), (' > ', ' >= '), (' >= ', ' > '), (' == ', ' != '), (' != ', ' == ')]),
    ('arithmetic', [(' + ', ' - '), (' - ', ' + '), (' * ', ' + ')]),
    ('boolean', [('return true', 'return false'), ('return false', 'return true')]),
    ('constant', [('= 0;', '= 1;')]),
]


def mutants(task):
    signature = task.java_reference[:-len(task.java_body)]
    out = []
    for group, rules in MUTATIONS:
        for old, new in rules:
            if old in task.java_body:
                out.append({'mutant_id': task.task_id + '#' + group, 'task_id': task.task_id, 'change': old.strip() + ' -> ' + new.strip(),
                            'java_code': signature + task.java_body.replace(old, new, 1)})
                break
    return out


def selfcheck(tasks, out, workers):
    """Stage 0: check the harness on known answers.
    - Reference solutions must compile, pass the tests and score CodeBLEU 1.0.
    - Wrapped in a ```java block as a model would answer, they must come out of the extraction unchanged.
    - Mutants of the references show how many wrong programs the tests catch."""
    os.makedirs(os.path.join(out, 'selfcheck'), exist_ok=True)
    by_id = {t.task_id: t for t in tasks}

    def reference(it):
        ref = by_id[it['task_id']].java_reference
        extracted = extract_java('Here is the code:\n```java\n%s\n```' % ref) == ref.strip() + '\n'
        return dict(_static_oracles(by_id[it['task_id']], ref), extraction_ok=extracted)

    path = os.path.join(out, 'selfcheck', 'reference.jsonl')
    items = _todo([{'task_id': t.task_id} for t in tasks], path, 'task_id')
    print('Reference solutions: %d' % len(items))
    _save_each(path, items, reference, workers)

    path = os.path.join(out, 'selfcheck', 'mutants.jsonl')
    items = _todo([m for t in tasks for m in mutants(t)], path, 'mutant_id')
    print('Mutants: %d' % len(items))
    _save_each(path, items, lambda m: dict(compile_check(by_id[m['task_id']], m['java_code']),
                                           **test_check(by_id[m['task_id']], m['java_code'])), workers)


def controls(model, tasks, out):
    """Stage 4: the judge on known answers: each reference solution (equivalent) and one mutant per task that
    the tests reject (not equivalent)."""
    by_id = {t.task_id: t for t in tasks}
    rejected = {}
    for m in read_jsonl(os.path.join(out, 'selfcheck', 'mutants.jsonl')):
        if m['task_id'] in by_id and m['compiles'] and m['tests'] != 'pass':
            rejected.setdefault(m['task_id'], m)
    items = [{'control_id': t.task_id + '#reference', 'task_id': t.task_id, 'kind': 'reference'} for t in tasks]
    items += [{'control_id': m['mutant_id'], 'task_id': m['task_id'], 'kind': 'mutant'} for m in rejected.values()]
    code = {it['control_id']: by_id[it['task_id']].java_reference for it in items if it['kind'] == 'reference'}
    code.update({m['mutant_id']: m['java_code'] for m in rejected.values()})
    path = os.path.join(out, 'controls.jsonl')
    items = _todo(items, path, 'control_id')
    print('Judge on %d known answers' % len(items))
    _save_each(path, items, lambda it: judge(model, by_id[it['task_id']], code[it['control_id']]))

Writing /kaggle/working/pilot/pipeline.py


In [7]:
%%writefile /kaggle/working/pilot/analyze.py
"""Combine the verdicts and measure how often the oracles disagree. The tests (O-c) are the reference, since
they are the only oracle that runs the translation.

Writes summary.md, summary.json, verdicts.csv (one row per task) and disagreements.csv (tasks where an oracle
disagrees with the tests, with an empty column for manual review)."""
import csv
import json
import os
from collections import Counter

from oracles import THRESHOLDS
from pipeline import read_jsonl

MAIN_THRESHOLD = 0.6


def pct(k, n):
    return round(100.0 * k / n, 1) if n else None


def against_tests(accepts, passes):
    pairs = list(zip(accepts, passes))
    tp, fp = sum(a and p for a, p in pairs), sum(a and not p for a, p in pairs)
    fn, tn = sum(p and not a for a, p in pairs), sum(not a and not p for a, p in pairs)
    return {'accepts_passes': tp, 'accepts_fails': fp, 'rejects_passes': fn, 'rejects_fails': tn,
            'agreement_pct': pct(tp + tn, len(pairs)), 'false_acceptance_pct': pct(fp, fp + tn),
            'false_rejection_pct': pct(fn, tp + fn)}


def correlation(xs, ys):
    n = len(xs)
    mx, my = sum(xs) / n, sum(ys) / n
    sxy = sum((x - mx) * (y - my) for x, y in zip(xs, ys))
    sxx, syy = sum((x - mx) ** 2 for x in xs), sum((y - my) ** 2 for y in ys)
    return round(sxy / (sxx * syy) ** 0.5, 3) if sxx and syy else None


def write_csv(path, rows):
    with open(path, 'w', newline='', encoding='utf-8') as f:
        if rows:
            w = csv.DictWriter(f, fieldnames=list(rows[0]))
            w.writeheader()
            w.writerows(rows)


def analyze(out):
    load = lambda name: {r['task_id']: r for r in read_jsonl(os.path.join(out, name))}
    trans, orc, jud = load('translations.jsonl'), load('oracles.jsonl'), load('judgments.jsonl')
    ids = [i for i in trans if i in orc]
    rows = [dict({'task_id': i, 'judge': jud.get(i, {}).get('judge', ''), 'seconds': trans[i]['seconds']},
                 **{k: orc[i][k] for k in ('compiles', 'tests', 'codebleu', 'cb_ngram', 'cb_weighted_ngram', 'cb_syntax', 'cb_dataflow')})
            for i in ids]
    n = len(rows)
    verdicts = {'O-b compile': [r['compiles'] for r in rows], 'O-c tests': [r['tests'] == 'pass' for r in rows]}
    for t in THRESHOLDS:
        verdicts['O-a CodeBLEU>=%.1f' % t] = [r['codebleu'] >= t for r in rows]
    if all(i in jud for i in ids):
        verdicts['O-h judge'] = [r['judge'] == 'YES' for r in rows]
    passes = verdicts['O-c tests']
    cb = [r['codebleu'] for r in rows]

    s = {'tasks': n, 'accept_pct': {k: pct(sum(v), n) for k, v in verdicts.items()}}
    s['compile_minus_tests_pp'] = round(s['accept_pct']['O-b compile'] - s['accept_pct']['O-c tests'], 1)
    s['test_status'] = dict(Counter(r['tests'] for r in rows))
    s['interface_errors'] = sum(r['compiles'] and r['tests'] == 'compile_error' for r in rows)
    s['judge_unparsed'] = sum(r['judge'] == 'UNPARSED' for r in rows)
    s['codebleu_mean_if_pass'] = round(sum(c for c, p in zip(cb, passes) if p) / max(1, sum(passes)), 3)
    s['codebleu_mean_if_fail'] = round(sum(c for c, p in zip(cb, passes) if not p) / max(1, n - sum(passes)), 3)
    s['codebleu_test_correlation'] = correlation(cb, [float(p) for p in passes])
    s['against_tests'] = {k: against_tests(v, passes) for k, v in verdicts.items() if k != 'O-c tests'}

    main = [k for k in verdicts if not k.startswith('O-a') or k.endswith('%.1f' % MAIN_THRESHOLD)]
    s['pairwise_agreement_pct'] = {a: {b: pct(sum(x == y for x, y in zip(verdicts[a], verdicts[b])), n) for b in main} for a in main}
    patterns = Counter(''.join('+' if verdicts[k][i] else '-' for k in main) for i in range(n))
    s['patterns'] = {'order': main, 'counts': dict(patterns.most_common())}

    ctrl = read_jsonl(os.path.join(out, 'controls.jsonl'))
    ref = [c['judge'] == 'YES' for c in ctrl if c['kind'] == 'reference']
    mut = [c['judge'] == 'NO' for c in ctrl if c['kind'] == 'mutant']
    if ref and mut:
        s['judge_on_known_answers'] = {'references': len(ref), 'yes_on_references': sum(ref),
                                       'mutants': len(mut), 'no_on_mutants': sum(mut),
                                       'balanced_accuracy_pct': round((pct(sum(ref), len(ref)) + pct(sum(mut), len(mut))) / 2, 1)}
    ref = read_jsonl(os.path.join(out, 'selfcheck', 'reference.jsonl'))
    mut = read_jsonl(os.path.join(out, 'selfcheck', 'mutants.jsonl'))
    if ref:
        s['selfcheck'] = {'references': len(ref), 'compile': sum(r['compiles'] for r in ref),
                          'pass_tests': sum(r['tests'] == 'pass' for r in ref),
                          'codebleu_1': sum(r['codebleu'] >= 0.9999 for r in ref), 'extraction_ok': sum(r['extraction_ok'] for r in ref),
                          'mutants': len(mut), 'mutants_compile': sum(m['compiles'] for m in mut),
                          'mutants_rejected': sum(m['compiles'] and m['tests'] != 'pass' for m in mut)}

    write_csv(os.path.join(out, 'verdicts.csv'), rows)
    disagreements = []
    for i, r in enumerate(rows):
        differ = [k.split()[0] for k in main if k != 'O-c tests' and verdicts[k][i] != passes[i]]
        if differ:
            disagreements.append(dict(r, differ_from_tests=' '.join(differ),
                                      tests_log=(orc[r['task_id']]['tests_log'].splitlines() or [''])[0][:200],
                                      judge_answer=jud.get(r['task_id'], {}).get('judge_answer', '')[:300].replace('\n', ' '),
                                      manual_review=''))
    write_csv(os.path.join(out, 'disagreements.csv'), disagreements)
    s['tasks_with_disagreement'] = len(disagreements)

    with open(os.path.join(out, 'summary.json'), 'w') as f:
        json.dump(s, f, indent=2)
    with open(os.path.join(out, 'summary.md'), 'w') as f:
        f.write(report(s))
    print(report(s))


def report(s):
    L = ['# Pilot results', '', '%d tasks.' % s['tasks'], '', '| Oracle | Accepts (%) |', '|---|---|']
    L += ['| %s | %s |' % kv for kv in s['accept_pct'].items()]
    L += ['', 'Compile rate minus test pass rate: %s percentage points. Test outcomes: %s. Interface errors: %d.'
          % (s['compile_minus_tests_pp'], s['test_status'], s['interface_errors']),
          'CodeBLEU: mean %s when the tests pass, %s when they fail; correlation with the test outcome %s.'
          % (s['codebleu_mean_if_pass'], s['codebleu_mean_if_fail'], s['codebleu_test_correlation']),
          '', '## Each oracle against the tests', '',
          '| Oracle | Accepts, passes | Accepts, fails | Rejects, passes | Rejects, fails | Agreement (%) | False acceptance (%) | False rejection (%) |',
          '|---|---|---|---|---|---|---|---|']
    L += ['| %s | %d | %d | %d | %d | %s | %s | %s |' % ((k,) + tuple(v.values())) for k, v in s['against_tests'].items()]
    names = list(s['pairwise_agreement_pct'])
    L += ['', '## Pairwise agreement (%)', '', '| | ' + ' | '.join(names) + ' |', '|---' * (len(names) + 1) + '|']
    L += ['| %s | ' % a + ' | '.join(str(v) for v in s['pairwise_agreement_pct'][a].values()) + ' |' for a in names]
    L += ['', '## Verdict patterns (%s; + accepts, - rejects)' % ', '.join(s['patterns']['order']), '', '| Pattern | Tasks |', '|---|---|']
    L += ['| `%s` | %d |' % kv for kv in s['patterns']['counts'].items()]
    L += ['', '%d tasks have at least one oracle that disagrees with the tests.' % s['tasks_with_disagreement']]
    if 'judge_on_known_answers' in s:
        j = s['judge_on_known_answers']
        L += ['', '## Judge on known answers', '', 'YES on %d of %d reference solutions; NO on %d of %d mutants rejected by the tests; balanced accuracy %s%%.'
              % (j['yes_on_references'], j['references'], j['no_on_mutants'], j['mutants'], j['balanced_accuracy_pct'])]
    if 'selfcheck' in s:
        c = s['selfcheck']
        L += ['', '## Self-check of the harness', '',
              'Reference solutions (%d): %d compile, %d pass the tests, %d score CodeBLEU 1.0, %d extracted unchanged.'
              % (c['references'], c['compile'], c['pass_tests'], c['codebleu_1'], c['extraction_ok']),
              'Mutants: %d generated, %d compile, %d rejected by the tests.' % (c['mutants'], c['mutants_compile'], c['mutants_rejected'])]
    return '\n'.join(L) + '\n'

Writing /kaggle/working/pilot/analyze.py


In [8]:
%%writefile /kaggle/working/pilot/run.py
"""Command line. Run from this folder, for example:

  python run.py all --translator qwen2.5-coder:7b-instruct-q4_K_M --judge llama3.1:8b-instruct-q4_K_M

Stages, in order: selfcheck, translate, check, judge, controls, analyze. 'all' runs them in turn; each can
also be run on its own. Models are Ollama model names."""
import argparse
import datetime
import json
import os
import platform
import subprocess
import sys

import pipeline
from analyze import analyze
from data import SHA256, load_tasks
from llm import Ollama, prompt_hashes

STAGES = ['selfcheck', 'translate', 'check', 'judge', 'controls', 'analyze']


def version(cmd):
    try:
        p = subprocess.run(cmd, capture_output=True, text=True)
        return [l for l in (p.stdout + p.stderr).splitlines() if not l.startswith('Picked up ')][0]
    except Exception:
        return None


def log_run(out, stage, models):
    path = os.path.join(out, 'run_info.json')
    info = json.load(open(path)) if os.path.exists(path) else {'stages': []}
    info['environment'] = {'python': platform.python_version(), 'java': version(['java', '-version']),
                           'gpu': version(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader']),
                           'dataset_sha256': SHA256, 'prompt_sha256': prompt_hashes()}
    info['stages'].append({'stage': stage, 'finished': datetime.datetime.now(datetime.timezone.utc).strftime('%Y-%m-%d %H:%M:%S UTC'),
                           'models': {k: m.info() for k, m in models.items()}})
    with open(path, 'w') as f:
        json.dump(info, f, indent=2)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('stage', choices=STAGES + ['all'])
    ap.add_argument('--translator', help='Ollama model that translates')
    ap.add_argument('--judge', help='Ollama model that judges (a different model)')
    ap.add_argument('--out', default='results')
    ap.add_argument('--limit', type=int, help='only the first N tasks, for a quick test')
    ap.add_argument('--workers', type=int, default=os.cpu_count(), help='parallel javac/java processes')
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tasks = load_tasks(limit=a.limit)
    for stage in STAGES if a.stage == 'all' else [a.stage]:
        print('\n== %s ==' % stage, flush=True)
        models = {}
        if stage == 'selfcheck':
            pipeline.selfcheck(tasks, a.out, a.workers)
        elif stage == 'translate':
            models['translator'] = Ollama(a.translator)
            pipeline.translate(models['translator'], tasks, a.out)
        elif stage == 'check':
            pipeline.check(tasks, a.out, a.workers)
        elif stage == 'judge':
            models['judge'] = Ollama(a.judge)
            pipeline.judge_all(models['judge'], tasks, a.out)
        elif stage == 'controls':
            models['judge'] = Ollama(a.judge)
            pipeline.controls(models['judge'], tasks, a.out)
        else:
            analyze(a.out)
        log_run(a.out, stage, models)


if __name__ == '__main__':
    sys.exit(main())

Writing /kaggle/working/pilot/run.py


## Run

In [9]:
import hashlib, json
os.chdir(CODE)
os.makedirs(OUT, exist_ok=True)
sha = {n: hashlib.sha256((open(n).read().rstrip('\n') + '\n').encode()).hexdigest() for n in sorted(os.listdir('.')) if n.endswith('.py')}
json.dump(sha, open(OUT + '/code_sha256.json', 'w'), indent=1)
LIM = '' if LIMIT is None else '--limit %d' % LIMIT
!python run.py selfcheck --out {OUT} {LIM}
!python run.py translate --translator {TRANSLATOR} --out {OUT} {LIM}
!python run.py check --out {OUT} {LIM}
!python run.py judge --judge {JUDGE} --out {OUT} {LIM}
!python run.py controls --judge {JUDGE} --out {OUT} {LIM}


== selfcheck ==
Reference solutions: 164
  20/164
  40/164
  60/164
  80/164
  100/164
  120/164
  140/164
  160/164
  164/164
Mutants: 298
  20/298
  40/298
  60/298
  80/298
  100/298
  120/298
  140/298
  160/298
  180/298
  200/298
  220/298
  240/298
  260/298
  280/298
  298/298

== translate ==
Translating 164 tasks with qwen2.5-coder:7b-instruct-q4_K_M
  20/164
  40/164
  60/164
  80/164
  100/164
  120/164
  140/164
  160/164
  164/164

== check ==
Compile, tests and CodeBLEU for 164 translations
  20/164
  40/164
  60/164
  80/164
  100/164
  120/164
  140/164
  160/164
  164/164

== judge ==
Judging 164 translations with llama3.1:8b-instruct-q4_K_M
  20/164
  40/164
  60/164
  80/164
  100/164
  120/164
  140/164
  160/164
  164/164

== controls ==
Judge on 306 known answers
  20/306
  40/306
  60/306
  80/306
  100/306
  120/306
  140/306
  160/306
  180/306
  200/306
  220/306
  240/306
  260/306
  280/306
  300/306
  306/306


## Results

In [10]:
!python run.py analyze --out {OUT} > /dev/null
import shutil
from IPython.display import Markdown
shutil.make_archive(OUT, 'zip', OUT)
server.terminate()
Markdown(open(OUT + '/summary.md').read())

# Pilot results

164 tasks.

| Oracle | Accepts (%) |
|---|---|
| O-b compile | 89.6 |
| O-c tests | 80.5 |
| O-a CodeBLEU>=0.5 | 82.3 |
| O-a CodeBLEU>=0.6 | 67.1 |
| O-a CodeBLEU>=0.7 | 51.8 |
| O-h judge | 73.8 |

Compile rate minus test pass rate: 9.1 percentage points. Test outcomes: {'pass': 132, 'compile_error': 18, 'fail': 14}. Interface errors: 1.
CodeBLEU: mean 0.756 when the tests pass, 0.507 when they fail; correlation with the test outcome 0.439.

## Each oracle against the tests

| Oracle | Accepts, passes | Accepts, fails | Rejects, passes | Rejects, fails | Agreement (%) | False acceptance (%) | False rejection (%) |
|---|---|---|---|---|---|---|---|
| O-b compile | 132 | 15 | 0 | 17 | 90.9 | 46.9 | 0.0 |
| O-a CodeBLEU>=0.5 | 118 | 17 | 14 | 15 | 81.1 | 53.1 | 10.6 |
| O-a CodeBLEU>=0.6 | 97 | 13 | 35 | 19 | 70.7 | 40.6 | 26.5 |
| O-a CodeBLEU>=0.7 | 80 | 5 | 52 | 27 | 65.2 | 15.6 | 39.4 |
| O-h judge | 105 | 16 | 27 | 16 | 73.8 | 50.0 | 20.5 |

## Pairwise agreement (%)

| | O-b compile | O-c tests | O-a CodeBLEU>=0.6 | O-h judge |
|---|---|---|---|---|
| O-b compile | 100.0 | 90.9 | 68.9 | 74.4 |
| O-c tests | 90.9 | 100.0 | 70.7 | 73.8 |
| O-a CodeBLEU>=0.6 | 68.9 | 70.7 | 100.0 | 68.9 |
| O-h judge | 74.4 | 73.8 | 68.9 | 100.0 |

## Verdict patterns (O-b compile, O-c tests, O-a CodeBLEU>=0.6, O-h judge; + accepts, - rejects)

| Pattern | Tasks |
|---|---|
| `++++` | 81 |
| `++-+` | 24 |
| `+++-` | 16 |
| `++--` | 11 |
| `----` | 7 |
| `--++` | 5 |
| `+---` | 5 |
| `+-++` | 4 |
| `+--+` | 4 |
| `---+` | 3 |
| `+-+-` | 2 |
| `--+-` | 2 |

76 tasks have at least one oracle that disagrees with the tests.

## Judge on known answers

YES on 97 of 164 reference solutions; NO on 117 of 142 mutants rejected by the tests; balanced accuracy 70.8%.

## Self-check of the harness

Reference solutions (164): 164 compile, 164 pass the tests, 164 score CodeBLEU 1.0, 164 extracted unchanged.
Mutants: 298 generated, 288 compile, 242 rejected by the tests.
